<left>
    <img src="https://weclouddata.s3.amazonaws.com/images/logos/wcd_logo_new_2.png" width='20%'>
</left>

----------

<h1 align="left"> Build a Research Paper Assistant using RAG</h1>
<center align="left"> <font size='4'>  Developed by: </font><font size='4' color='#33AAFBD'>WeCloudData</font></center>
<br>

----------

In [ ]:
!pip install faiss-cpu sentence-transformers transformers datasets==2.14.5


INFO: pip is looking at multiple versions of transformers to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of transformers to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.6 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 519.6/519.6 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 51.6 MB/s

**Import Libraries**

In [ ]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from transformers import pipeline
import faiss
import numpy as np
import pandas as pd


**Load and Explore Research Data**

**Scenario**

You are building an assistant for AIResearchHub, a company that wants to summarize or answer questions about new computer-science papers.

In [ ]:
from datasets import load_dataset

# Use scientific_papers (arXiv split) and take a small subset
dataset = load_dataset("armanc/scientific_papers", "arxiv", split="train[:100]")

import pandas as pd
df = pd.DataFrame(dataset)[["abstract"]]  # sometimes “title” might be absent, so we focus on abstract
df.head()


Extracting data files:   0%|          | 0/2 [00:00<?, ?it/s]

Generating train split:   0%|          | 0/203037 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/6436 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6440 [00:00<?, ? examples/s]

,abstract
0,additive models play an important role in sem...
1,"we have studied the leptonic decay @xmath0 , ..."
2,"in 84 , 258 ( 2000 ) , mateos conjectured tha..."
3,the effect of a random phase diffuser on fluc...
4,with a special intention of clarifying the un...


In [ ]:
print("Number of papers:", len(df))
print("\nSample abstract:\n")
print(df['abstract'][0][:500], "...")


Number of papers: 100

Sample abstract:

 additive models play an important role in semiparametric statistics . 
 this paper gives learning rates for regularized kernel based methods for additive models . 
 these learning rates compare favourably in particular in high dimensions to recent results on optimal learning rates for purely nonparametric regularized kernel based quantile regression using the gaussian radial basis function kernel , provided the assumption of an additive model is valid . 
 additionally , a concrete example is pr ...


**Create Document Embeddings**

In [ ]:
embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

documents = df['abstract'].tolist()
doc_embeddings = embedding_model.encode(documents, convert_to_numpy=True, normalize_embeddings=True)
print("Embeddings shape:", doc_embeddings.shape)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings shape: (100, 384)


**Build a FAISS Index for Retrieval**

In [ ]:
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(doc_embeddings)
print(f"FAISS index contains {index.ntotal} abstracts.")


FAISS index contains 100 abstracts.


**Retrieval Function**

In [ ]:
def retrieve(query, top_k=3):
    query_emb = embedding_model.encode([query], normalize_embeddings=True)
    distances, indices = index.search(query_emb, top_k)
    retrieved_docs = [documents[i] for i in indices[0]]
    return retrieved_docs


**Test**

In [ ]:
query = "What are recent advancements in reinforcement learning?"
results = retrieve(query)

for i, abs_text in enumerate(results, start=1):
    print(f"\n Abstract {i}:\n{abs_text[:400]}...")



 Abstract 1:
 synaptic memory is considered to be the main element responsible for learning and cognition in humans . 
 although traditionally non - volatile long - term plasticity changes have been implemented in nanoelectronic synapses for neuromorphic applications , recent studies in neuroscience have revealed that biological synapses undergo meta - stable volatile strengthening followed by a long - term st...

 Abstract 2:
 additive models play an important role in semiparametric statistics . 
 this paper gives learning rates for regularized kernel based methods for additive models . 
 these learning rates compare favourably in particular in high dimensions to recent results on optimal learning rates for purely nonparametric regularized kernel based quantile regression using the gaussian radial basis function kernel...

 Abstract 3:
 we propose a tuner , suitable for adaptive control and ( in its discrete - time version ) adaptive filtering applications , that sets the second deri

**Add a Generation Model (FLAN-T5)**

In [ ]:
generator = pipeline("text2text-generation", model="google/flan-t5-base")


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Device set to use cpu


**Define RAG Pipeline**

In [ ]:
def rag_pipeline(query):
    retrieved_docs = retrieve(query)
    context = " ".join(retrieved_docs)
    prompt = f"Context: {context}\nQuestion: {query}\nAnswer:"

    response = generator(prompt, max_length=120, temperature=0.3)[0]['generated_text']

    print("🔍 Retrieved Abstracts:")
    for i, doc in enumerate(retrieved_docs, start=1):
        print(f" - Abstract {i}: {doc[:120]}...")

    print("\n💬 Generated Answer:\n", response)


**Ask A question**

In [ ]:
rag_pipeline("What are the applications of graph neural networks?")


Token indices sequence length is longer than the specified maximum sequence length for this model (793 > 512). Running this sequence through the model will result in indexing errors
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Both `max_new_tokens` (=256) and `max_length`(=120) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🔍 Retrieved Abstracts:
 - Abstract 1:  phylogenetic networks are mathematical structures for modeling and visualization of reticulation processes in the study...
 - Abstract 2:  synaptic memory is considered to be the main element responsible for learning and cognition in humans . 
 although trad...
 - Abstract 3:  * _ abstract _ : * in this paper , we present a new pipeline which automatically identifies and annotates axoplasmic re...

💬 Generated Answer:
 phylogenetic networks are mathematical structures for modeling and visualization of reticulation processes in the study of evolution
